# MusicSage — STFT модели vs идеальные стемы

Что здесь происходит:

1. Загружается обученная модель **HybridUNet** (`model.py`, чекпойнт `checkpoints_pytorch/sep_best.pt`)
   — 1D waveform U-Net (Demucs-стайл) + 2D спектральная ветка с масками, как в `separator_pytorch.ipynb`;
2. Кусок трека режется на перекрывающиеся чанки с Hann-кросфейдом, и модель предсказывает 4 стема
   (drums / bass / other / vocals) — ровно как инференс в ноутбуке-трейнере;
3. Для **предсказанных** и **идеальных** (ground-truth) стемов считаются STFT с настраиваемыми
   параметрами и рисуются спектрограммы рядом + карта разницы в дБ;
4. Всё настраивается в ячейке `CONFIG` ниже:
   - `START_SEC` / `DISPLAY_SEC` — какой кусок трека показать (в секундах);
   - `VIS_NFFT` / `VIS_HOP` / `VIS_WINDOW` — настройки STFT для графиков;
   - `VIS_FMAX_HZ` / `VIS_DB_FLOOR` — диапазон по частоте и динамический диапазон дБ;
   - `DATA_SOURCE` — `"musdb"` (тестовый трек MUSDB18, идеальные стемы есть)
     или `"file"` (свой аудиофайл; идеальные стемы опционально из `REF_DIR`);
   - `USE_WIENER` — применять ли Wiener soft-mask пост-процессинг после модели.


In [ ]:
# ============================================================
# Импорты и фикс stempeg
# ============================================================
import os
import warnings

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

# фикс бага stempeg (см. separator_pytorch.ipynb)
if not hasattr(warnings, "warning"):
    warnings.warning = warnings.warn
warnings.filterwarnings("ignore", message="Stems differ in length")


In [ ]:
# ============================================================
# CONFIG — настройки отображения
# ============================================================

# ---- какой кусок трека показать ----
START_SEC = 30.0          # с какой секунды трека начать
DISPLAY_SEC = 6.0         # сколько секунд показать (можно весь трек)

# ---- какие стемы показать ----
SOURCES_TO_PLOT = ["drums", "bass", "other", "vocals"]

# ---- настройки STFT для графиков (меняются свободно) ----
VIS_NFFT = 2048           # размер окна FFT
VIS_HOP = 512             # шаг между кадрами
VIS_WINDOW = "sqrt_hann"  # тип окна: "hann" | "sqrt_hann" | "blackman"
VIS_FMAX_HZ = 16000       # верхняя граница частоты на графике, Гц
VIS_DB_FLOOR = 90         # динамический диапазон спектрограммы, дБ

# ---- модель / данные ----
CKPT_PATH = "checkpoints_pytorch/sep_best.pt"

DATA_SOURCE = "musdb"     # "musdb" — трек из MUSDB18 (есть идеальные стемы)
                         # "file"  — свой аудиофайл (USER_PATH)
DB_ROOT = "musdb18/wav"
TRACK_INDEX = 0           # индекс трека в тестовой выборке musdb
USER_PATH = "my_song.mp3" # путь к своему файлу (при DATA_SOURCE="file")
REF_DIR = None            # при DATA_SOURCE="file": папка с drums/bass/other/vocals.wav
                         # (идеальные стемы для сравнения, опционально)

USE_WIENER = False        # True: Wiener soft-mask пост-процессинг (как в ноутбуке-трейнере)


In [ ]:
# ============================================================
# Загрузка модели
# ============================================================
import sys
sys.path.insert(0, ".")

from model import HybridUNet, SOURCE_NAMES, SAMPLE_RATE, N_SOURCES

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", DEVICE)

# Сверхсетка WaveUNet: вход паддится до кратного 4**LEVELS
STRIDE = 4
LEVELS = 5
CHUNK_SAMPLES = SAMPLE_RATE * 6                 # чанк как на трейнинге (6 с)
PAD_LEN = ((CHUNK_SAMPLES + STRIDE ** LEVELS - 1) // STRIDE ** LEVELS) * STRIDE ** LEVELS

model = HybridUNet(n_sources=N_SOURCES).to(DEVICE)

state = torch.load(CKPT_PATH, map_location="cpu")
if isinstance(state, dict) and any(k.startswith("module.") for k in state):
    state = {k[len("module."):]: v for k, v in state.items()}   # DataParallel -> чистые имена
model.load_state_dict(state)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {CKPT_PATH} | params: {n_params:,}")

# sanity-прогон: вход (1, 2, N) -> выход (1, 4, 2, N)
with torch.no_grad():
    out = model(torch.zeros(1, 2, PAD_LEN, device=DEVICE))
print("input:", (1, 2, PAD_LEN), "-> output:", tuple(out.shape))


In [ ]:
# ============================================================
# Загрузка данных: MUSDB18 или свой файл
# ============================================================
import musdb
import stempeg


def load_audio(path, sr=SAMPLE_RATE):
    """Любой аудиофайл (mp3/wav/flac/...) -> стерео float32 (N, 2) @ sr.
    Моно-файлы дублируются в оба канала."""
    stems, _ = stempeg.read_stems(path, sample_rate=sr, ffmpeg_format="s16le")
    audio = np.squeeze(stems)
    if audio.ndim == 1:
        audio = np.stack([audio, audio], axis=1)
    elif audio.shape[1] != 2:
        audio = audio.mean(axis=1)
        audio = np.stack([audio, audio], axis=1)
    return audio.astype(np.float32)


if DATA_SOURCE == "musdb":
    if not os.path.isdir(os.path.join(DB_ROOT, "test")):
        raise FileNotFoundError(
            f"MUSDB18 не найден: {DB_ROOT}. Переключи DATA_SOURCE='file' "
            "и укажи USER_PATH."
        )
    mus_test = musdb.DB(root=DB_ROOT, subsets="test", sample_rate=SAMPLE_RATE)
    track = mus_test[TRACK_INDEX]
    mix = track.audio.astype(np.float32)                 # (N, 2)
    ref_stems = np.stack(
        [track.targets[name].audio.astype(np.float32) for name in SOURCE_NAMES]
    )                                                    # (4, N, 2) идеальные стемы
    title_info = track.name
    print("source: MUSDB18 (test) | track:", track.name)
else:
    track = None
    if not os.path.exists(USER_PATH):
        raise FileNotFoundError(f"файл не найден: {USER_PATH}")
    mix = load_audio(USER_PATH)                          # (N, 2)
    title_info = USER_PATH
    ref_stems = None
    if REF_DIR:
        refs = []
        for name in SOURCE_NAMES:
            p = os.path.join(REF_DIR, f"{name}.wav")
            if not os.path.exists(p):
                print(f"референс не найден: {p} — пропускаю")
                refs = None
                break
            r = load_audio(p)
            if len(r) < len(mix):
                r = np.pad(r, ((0, len(mix) - len(r)), (0, 0)))
            refs.append(r[:len(mix)])
        if refs is not None:
            ref_stems = np.stack(refs)
    print("source: file |", title_info)

print("duration:", f"{len(mix) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz")
print("stems :", SOURCE_NAMES)
print("refs  :", "есть (ground truth)" if ref_stems is not None else "нет")


In [ ]:
# ============================================================
# Инференс: кусок трека -> 4 стема (перекрывающиеся чанки + Hann-кросфейд)
# ============================================================
def wiener_postprocess(stems, mix, gamma=2.0, fl=4096, fs=1024, seg=30 * SAMPLE_RATE):
    """Wiener soft-mask пост-процессинг: маски по |STFT(стem)|^gamma,
    итог = ISTFT(mask * STFT(mix)) — сумма стемов точно равна миксу.
    stems: (N, 4, 2), mix: (N, 2) -> (N, 4, 2)."""
    from model import stft as model_stft, get_window as model_get_window
    out = np.empty_like(stems)
    for start in range(0, len(mix), seg):
        e = stems[start:start + seg]
        m = mix[start:start + seg]
        s = model_stft(torch.from_numpy(np.ascontiguousarray(e.transpose((1, 2, 0)))).to(DEVICE),
                       fl, fs)                            # (4, 2, F, T)
        mst = model_stft(torch.from_numpy(np.ascontiguousarray(m.transpose((1, 0)))).to(DEVICE),
                         fl, fs)                          # (2, F, T)
        mag = s.abs().pow(gamma)
        masks = mag / (mag.sum(dim=0, keepdim=True) + 1e-4)
        est = (masks * mst.unsqueeze(0)).reshape(-1, *mst.shape[-2:])
        r = torch.istft(est, fl, fs, window=model_get_window(fl, DEVICE), length=len(m))
        out[start:start + seg] = r.reshape(4, 2, len(m)).permute(2, 0, 1).cpu().numpy()
    return out


def separate_window(model, mix, device, use_wiener=False):
    """Кусок микса (N, 2) -> (N, 4, 2) стерео-стемы.
    Чанки по CHUNK_SAMPLES с 50% overlap и Hann-кросфейдом; нормализация
    per-chunk по std микса — ровно как на трейнинге."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2
    out = np.zeros((n, 4, 2), dtype=np.float32)
    weights = np.zeros((n, 2), dtype=np.float32)
    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)
            scale = x.std(dim=0).clamp_min(1e-4)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))       # (1, 2, N+pad)
            est = model(xp)[:, :, :, :CHUNK_SAMPLES] * scale[:, None]
            est = est.squeeze(0).cpu().numpy().transpose(2, 0, 1)  # (N, 4, 2)
            w = np.hanning(CHUNK_SAMPLES).astype(np.float32)
            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None, None]
            weights[st:stop] += w[:stop - st, None]
    ok = weights[:, 0] > 0
    out[ok] /= weights[ok, None]
    if use_wiener:
        out = wiener_postprocess(out, mix)
    return out


In [ ]:
# ============================================================
# Вырезаем окно, гоняем модель
# ============================================================
start = int(START_SEC * SAMPLE_RATE)
disp = int(DISPLAY_SEC * SAMPLE_RATE)
max_start = max(0, len(mix) - disp)          # чтобы окно не оказалось пустым
start = min(start, max_start)
stop = min(start + disp, len(mix))

mix_win = mix[start:stop]
ref_win = ref_stems[:, start:stop] if ref_stems is not None else None   # (4, N, 2)
pred_win = separate_window(model, mix_win, DEVICE, use_wiener=USE_WIENER)

print(f"window: {start / SAMPLE_RATE:.1f}s .. {stop / SAMPLE_RATE:.1f}s"
      f" ({len(mix_win) / SAMPLE_RATE:.2f}s, {len(mix_win)} samples)")
print("pred shape:", pred_win.shape)


In [ ]:
# ============================================================
# STFT c настраиваемыми параметрами (через torch)
# ============================================================
_WINDOWS = {}


def get_window(n_fft, device, kind):
    key = (n_fft, str(device), kind)
    if key not in _WINDOWS:
        if kind == "sqrt_hann":
            w = torch.sqrt(torch.hann_window(n_fft, periodic=True, device=device))
        elif kind == "hann":
            w = torch.hann_window(n_fft, periodic=True, device=device)
        elif kind == "blackman":
            w = torch.blackman_window(n_fft, periodic=True, device=device)
        else:
            raise ValueError(kind)
        _WINDOWS[key] = w
    return _WINDOWS[key]


def compute_stft(audio, n_fft, hop, window_kind, device):
    """Моно (N,) -> комплексный STFT (F, T)."""
    x = torch.from_numpy(audio.astype(np.float32)).to(device)
    return torch.stft(x, n_fft=n_fft, hop_length=hop, win_length=n_fft,
                      window=get_window(n_fft, device, window_kind),
                      return_complex=True)


def to_mono(stereo):
    """(N, 2) -> (N,) — для спектрограммы берём среднее по каналам."""
    return stereo.mean(axis=1)


stfts = {}
for name in SOURCES_TO_PLOT:
    i = SOURCE_NAMES.index(name)
    stfts[name] = {"pred": compute_stft(to_mono(pred_win[:, i]), VIS_NFFT, VIS_HOP, VIS_WINDOW, DEVICE)}
    if ref_win is not None:
        stfts[name]["ref"] = compute_stft(to_mono(ref_win[i]), VIS_NFFT, VIS_HOP, VIS_WINDOW, DEVICE)
print("STFT done | n_fft:", VIS_NFFT, "| hop:", VIS_HOP, "| window:", VIS_WINDOW)
for name in SOURCES_TO_PLOT:
    print(f"  {name:>6s}: frames {tuple(stfts[name]['pred'].shape)}")


In [ ]:
# ============================================================
# SI-SDR по каждому стему на выбранном окне (если есть идеальные стемы)
# ============================================================
EPS = 1e-8


def si_sdr(estimate, reference):
    """SI-SDR (дБ): (N,) или (N, 2) -> среднее по каналам."""
    e = estimate - estimate.mean(axis=0, keepdims=True)
    r = reference - reference.mean(axis=0, keepdims=True)
    alpha = np.sum(r * e, axis=0) / (np.sum(r * r, axis=0) + EPS)
    dist = e - alpha * r
    return float((10 * np.log10(alpha ** 2 * np.sum(r * r, axis=0)
                               / (np.sum(dist * dist, axis=0) + EPS))).mean())


sdr_vals = {}
if ref_win is not None:
    for name in SOURCES_TO_PLOT:
        i = SOURCE_NAMES.index(name)
        sdr_vals[name] = si_sdr(pred_win[:, i], ref_win[i])
        print(f"SI-SDR ({name:>6s}): {sdr_vals[name]:.2f} dB")
else:
    print("идеальные стемы недоступны — SI-SDR не считаю")


In [ ]:
# ============================================================
# Графики: предсказанный STFT vs идеальный vs разница
# ============================================================
def spec_db(spec):
    """Комплексный STFT -> log-магнитуда в дБ."""
    return 20.0 * np.log10(spec.abs().cpu().numpy() + 1e-8)


def show_spec(ax, spec_db_vals, sr, n_fft, hop, vmin, vmax):
    F = spec_db_vals.shape[0]
    fmax_bin = min(F, int(VIS_FMAX_HZ * n_fft / sr))
    img = ax.imshow(spec_db_vals[:fmax_bin], origin="lower", aspect="auto",
                    cmap="magma", vmin=vmin, vmax=vmax,
                    extent=[0, spec_db_vals.shape[1] * hop / sr, 0, fmax_bin * sr / n_fft])
    ax.set_xlabel("time, s")
    ax.set_ylabel("freq, Hz")
    return img


has_ref = ref_win is not None
n_cols = 3 if has_ref else 1
n_src = len(SOURCES_TO_PLOT)
fig, axes = plt.subplots(n_src, n_cols, figsize=(17 if has_ref else 7, 4.0 * n_src))
axes = np.atleast_2d(axes).reshape(n_src, n_cols)

for row, name in enumerate(SOURCES_TO_PLOT):
    pred_db = spec_db(stfts[name]["pred"])
    ref_db = spec_db(stfts[name]["ref"]) if has_ref else None
    vmax = max(pred_db.max(), ref_db.max()) if has_ref else pred_db.max()
    vmin = vmax - VIS_DB_FLOOR

    sdr_txt = f" (SI-SDR {sdr_vals[name]:.1f} dB)" if has_ref else ""
    img = show_spec(axes[row, 0], pred_db, SAMPLE_RATE, VIS_NFFT, VIS_HOP, vmin, vmax)
    axes[row, 0].set_title(f"{name} — модель{sdr_txt}", fontsize=11)
    fig.colorbar(img, ax=axes[row, 0], fraction=0.046)

    if has_ref:
        img = show_spec(axes[row, 1], ref_db, SAMPLE_RATE, VIS_NFFT, VIS_HOP, vmin, vmax)
        axes[row, 1].set_title(f"{name} — идеал", fontsize=11)
        fig.colorbar(img, ax=axes[row, 1], fraction=0.046)

        diff_db = 20.0 * np.log10((stfts[name]["pred"].abs()
                                  / stfts[name]["ref"].abs().clamp_min(1e-4)
                                  ).cpu().numpy() + 1e-8)
        diff_db = np.clip(diff_db, -30, 30)
        img = show_spec(axes[row, 2], diff_db, SAMPLE_RATE, VIS_NFFT, VIS_HOP, -30, 30)
        axes[row, 2].set_title(f"{name} — разница |pred|-|ref|, дБ", fontsize=11)
        img.set_cmap("RdBu_r")
        fig.colorbar(img, ax=axes[row, 2], fraction=0.046)

fig.suptitle(f"STFT {VIS_NFFT}/{VIS_HOP} ({VIS_WINDOW}) | {title_info} | "
             f"{start / SAMPLE_RATE:.1f}s..{stop / SAMPLE_RATE:.1f}s", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()


In [ ]:
# ============================================================
# Сохранить предсказанные стемы в wav
# ============================================================
import soundfile as sf

os.makedirs("output", exist_ok=True)
base = os.path.splitext(os.path.basename(title_info))[0]
for i, name in enumerate(SOURCE_NAMES):
    sf.write(f"output/{base}_{name}.wav", pred_win[:, i], SAMPLE_RATE)
sf.write(f"output/{base}_mix.wav", mix_win, SAMPLE_RATE)
print("saved output/*.wav")
